# 🚀 Eksperimen Alternatif 3: Project-Agnostic Feature Engineering & Normalisasi Relatif
## Mitigasi Bias Skala Proyek pada Cross-Project Flaky Test Prediction

- **Kelompok Peneliti**: Kelompok A01 — Rekayasa Perangkat Lunak A (Magister Teknik Informatika ITS)
- **Hardware Akselerasi**: Laptop GPU NVIDIA GeForce RTX 4050 (6GB GDDR6 VRAM, CUDA 13.4, Driver 617.42)
- **Dataset**: FlakeFlagger (22.236 test cases dari 24 repositori Java, C1 Row-Level Drop: 21.963 baris)
- **Rujukan Paper Utama**:
  1. Alshammari et al. (ICSE 2021 — *FlakeFlagger: Predicting Flakiness Without Rerunning Tests*)
  2. Afeltra et al. (IEEE Access 2024 — *A Large-Scale Empirical Investigation Into Cross-Project Flaky Test Prediction*)
  3. Pontillo et al. (EMSE 2022 — *Static Test Flakiness Prediction: How Far Can We Go?*)

### 1. Verifikasi Akselerasi GPU NVIDIA RTX 4050 & CUDA 13.4

In [ ]:
import sys
from pathlib import Path

# Tambahkan path src ke sys.path
src_path = Path("../src").resolve()
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from check_gpu import check_nvidia_smi, check_python_environment, check_xgboost_gpu
check_nvidia_smi()
check_python_environment()
check_xgboost_gpu()

### 2. Pemuatan & Pemeriksaan Dataset FlakeFlagger (C1: Row-Level Drop)

In [ ]:
from data_loader import load_dataset

df, features, projects = load_dataset("C1")
print(f"Total baris valid: {len(df):,}")
print(f"Total fitur asli FlakeFlagger: {len(features)}")
print(f"Jumlah proyek: {len(projects)}")
print(f"Total flaky tests: {df['flaky'].sum():,} ({df['flaky'].mean()*100:.2f}%)")

### 3. Demonstrasi Pipeline Rekayasa Fitur Project-Agnostic & PCA Churn

In [ ]:
import numpy as np
import pandas as pd
from feature_engineer import FeatureSetPipeline, compute_vif_dataframe, CHURN_HINDEX_RAW

pipeline = FeatureSetPipeline(k_features=12, pca_variance_threshold=0.95)
pipeline.fit(df, df["flaky"])
transformed_all = pipeline.transform(df)

print("Ringkasan Dimensi Ruang Fitur:")
for set_name, set_df in transformed_all.items():
    print(f"- {set_name:<22}: {set_df.shape[1]} fitur")

print("\nKomponen PCA Churn Terpilih:", pipeline.pca_transformer.component_names_)
print("Explained Variance Ratio per Komponen:", np.round(pipeline.pca_transformer.explained_variance_ratio_, 4))
print(f"Kumulatif Variansi: {np.sum(pipeline.pca_transformer.explained_variance_ratio_)*100:.2f}%")
print("Fitur F4 Terseleksi via ANOVA F-test:", pipeline.selected_f4_feature_names)

### 4. Analisis Multikolinearitas (VIF) Sebelum vs Sesudah PCA

In [ ]:
vif_raw = compute_vif_dataframe(df[CHURN_HINDEX_RAW])
pca_cols = pipeline.pca_transformer.component_names_
vif_pca = compute_vif_dataframe(transformed_all["F3_Project_Agnostic"][pca_cols])

print("--- Top VIF Raw Churn (hIndex Windows) ---")
print(vif_raw)

print("\n--- VIF PCA Churn Components (Orthogonal) ---")
print(vif_pca)

### 5. Eksekusi Penuh LOPO-CV (Leave-One-Project-Out CV) Lintas 24 Proyek
Membandingkan Set Fitur F1 (Original), F2 (Ratios Only), F3 (Project-Agnostic Reformed), dan F4 (Selected F3) pada XGBoost GPU dan Random Forest.

In [ ]:
from lopo_feature_runner import run_lopo_feature_engineering

# Jalankan LOPO-CV pada seluruh 24 proyek
df_folds, summary_df = run_lopo_feature_engineering()

### 6. Tabel Ringkasan Metrik Komparasi (Mean ± Std Dev)

In [ ]:
display_cols = ["Model", "Feature_Set", "N_Features", "PR_AUC_Mean", "PR_AUC_Std", "F1_Mean", "F1_Std", "Recall_Mean", "Precision_Mean", "Train_Time_Avg_Sec"]
summary_df[display_cols]

### 7. Hasil Uji Signifikansi Statistik Wilcoxon & Cliff's Delta

In [ ]:
import pandas as pd
wilcox_path = Path("../results/wilcoxon_test_results.csv").resolve()
if wilcox_path.exists():
    df_wilcox = pd.read_csv(wilcox_path)
    display(df_wilcox[(df_wilcox["Metric"].isin(["PR_AUC", "F1"])) & (df_wilcox["Model"] == "xgboost_gpu")])
else:
    print("File hasil Wilcoxon belum ditemukan.")

### 8. Galeri Visualisasi Hasil Eksperimen Publikasi

In [ ]:
from IPython.display import Image, display
plots_dir = Path("../results/plots").resolve()

plot_files = [
    "correlation_heatmap_pca.png",
    "pca_scree_variance.png",
    "metric_boxplots_xgboost.png",
    "summary_performance_bars.png",
    "feature_importance_f3.png",
    "vif_comparison_bars.png"
]

for pf in plot_files:
    p = plots_dir / pf
    if p.exists():
        print(f"=== Visualisasi: {pf} ===")
        display(Image(filename=str(p)))
    else:
        print(f"Grafik {pf} belum dibuat.")